In [1]:
# %% [markdown]
# # Train DeepBDE with NSPPK-augmented features
#
# This notebook concatenates NSPPK features onto the global feature vector of the DeepBDE
# reaction graph, then trains a fresh model from scratch.
#
# **How the concatenation works:**
# - The DeepBDE reaction graph has three node types: atom, bond, and global.
# - The global node normally carries 3 scalar features (num_atoms, num_bonds, mol_weight).
# - `GlobalFeaturize` (in architecture/data/featurizers.py) already supports appending NSPPK
#   features. We just need to pass `nsppk_params` when building the dataset.
# - The resulting global feature size is `3 + 2**nbits`.
# - We tell the model about this via `in_feat_sizes={'atom': ..., 'bond': 7, 'global': 3 + 2**nbits}`.

# %% [markdown]

In [2]:
pip install dgl

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install torchdata==0.9.0

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install PyYAML

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 4.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install pydantic

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 2.7 MB/s  0:00:00 eta 0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pydantic]3/4 [pydantic]core]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install torch==2.4.1 torchvision torchaudio

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.7 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
IN

In [8]:
# ## 0. Imports

# %%
import sys, os, pickle, math, logging, csv

# Make sure imports resolve from the GNNpredict root
#ROOT = os.path.dirname(os.path.abspath(__file__))
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
from torch.nn import MSELoss
from torch import nn`
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import (
    mean_absolute_error, mean_absolute_percentage_error,
    max_error, root_mean_squared_error, r2_score,
)

from architecture import construct_model
from architecture.data.dataset import BDEDataset, BDESubset
from architecture.data.dataloader import RxnDataLoader
from architecture.data.dset_generate import get_atomic_num_list
from architecture.data.featurizers import AtomFeaturize, BondFeaturize, GlobalFeaturize
from architecture.data.initial_containers import DirectSmilesRepo, DGLwBDEMappings
from architecture.item_handle import deep_bde_item_handle
from train.trainer import Trainer
from train.test.test_on_set import TestonSet

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")


FileNotFoundError: Cannot find DGL C++ graphbolt library at /home/max/miniconda3/envs/bde/lib/python3.11/site-packages/dgl/graphbolt/libgraphbolt_pytorch_2.8.0.so